# Part 3: Prompt Engineering & Evaluation — SOC Edition

In [1]:
# --- Reconnect to Foundry (run this first after opening a new notebook) ---
import subprocess, sys, json, shutil
from pathlib import Path

def run_cmd(args, check=True):
    exe = shutil.which(args[0])
    if exe: args = [exe] + args[1:]
    result = subprocess.run(args, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    return result

# Auto-detect pre-provisioned environment
config_path = Path.cwd().parent / "workshop_config.json"
if not config_path.exists():
    config_path = Path.cwd() / "workshop_config.json"
if config_path.exists():
    _cfg = json.loads(config_path.read_text())
    PROJECT_ENDPOINT = _cfg["PROJECT_ENDPOINT"]
    MODEL_DEPLOYMENT_NAME = _cfg["MODEL_DEPLOYMENT_NAME"]
else:
    raise FileNotFoundError(
        "workshop_config.json not found. Run 00-setup.ipynb first."
    )

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FileSearchTool, WebSearchPreviewTool, FunctionTool
from azure.identity import DefaultAzureCredential, AzureCliCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()
print(f"✅ Connected to: {PROJECT_ENDPOINT}")
print(f"   Model: {MODEL_DEPLOYMENT_NAME}")

✅ Connected to: https://fndryiac2ttkx3.services.ai.azure.com/api/projects/iac-ops-demo
   Model: gpt-4.1-mini


---
## Section 3: Prompt Engineering (~7 min)

The quality of an agent depends heavily on its **instructions** (system prompt).
Key patterns:
- **Persona**: Define who the agent is and its expertise
- **Constraints**: What it should and shouldn't do
- **Output format**: Specify structure (JSON, markdown, etc.)
- **Few-shot examples**: Show desired input/output pairs

In [2]:
# --- 3.1 Basic vs. Engineered Prompt ---
# Compare a vague prompt with a well-engineered one for SOC alert triage.

QUESTION = "Analyze this alert: user john.doe@contoso.com signed in from Seattle at 08:30 and Lagos at 08:47. Is this a compromise?"

# Basic prompt
basic_agent = project_client.agents.create_version(
    agent_name="03-soc-basic-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions="You are helpful.",
    ),
)
r_basic = openai_client.responses.create(
    input=QUESTION,
    extra_body={"agent_reference": {"name": basic_agent.name, "type": "agent_reference"}},
)

# Engineered prompt
eng_agent = project_client.agents.create_version(
    agent_name="03-soc-engineered-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a senior SOC L2 analyst with 10 years of experience in "
            "identity threat detection and Microsoft Sentinel investigations.\n\n"
            "RULES:\n"
            "- Structure your analysis with clear headings: Summary, Risk Factors, Verdict, Recommended Actions\n"
            "- Always calculate the physical distance and required travel velocity between locations\n"
            "- Reference MITRE ATT&CK techniques where applicable\n"
            "- Assess authentication method, device compliance, and IP reputation\n"
            "- Provide a PACO verdict: AccountCompromised, LikelyCompromised, Suspicious, BenignAnomaly, or InsufficientData\n"
            "- Keep responses under 400 words\n\n"
            "FORMAT: Use markdown with ## headings."
        ),
    ),
)
r_eng = openai_client.responses.create(
    input=QUESTION,
    extra_body={"agent_reference": {"name": eng_agent.name, "type": "agent_reference"}},
)

print("=" * 60)
print("BASIC PROMPT RESPONSE:")
print("=" * 60)
print(r_basic.output_text[:500])
print(f"\n... ({len(r_basic.output_text)} chars total)")
print("\n" + "=" * 60)
print("ENGINEERED PROMPT RESPONSE:")
print("=" * 60)
print(r_eng.output_text[:800])
print(f"\n... ({len(r_eng.output_text)} chars total)")

BASIC PROMPT RESPONSE:
This alert indicates that the user john.doe@contoso.com signed in from two geographically distant locations within a very short time span:

- Seattle at 08:30  
- Lagos at 08:47  

Given the distance between Seattle (USA) and Lagos (Nigeria) is roughly 13,000 kilometers, it is physically impossible for the user to travel between these two locations within 17 minutes.

### Analysis:
- This behavior is known as **impossible travel** or **impossible travel activity**, which is a common indicator of

... (1272 chars total)

ENGINEERED PROMPT RESPONSE:
## Summary  
User john.doe@contoso.com logged in from Seattle at 08:30 and then from Lagos at 08:47 on the same day. These are two geographically distant locations within 17 minutes.

## Risk Factors  
- **Geographical distance:** Seattle, WA, USA to Lagos, Nigeria is approximately 13,500 km (8,390 miles) apart.  
- **Time difference:** 17 minutes between logins.  
- **Required velocity:** To travel 13,500 km in 17 minu

In [3]:
# --- 3.2 Structured JSON Output ---
# Instruct the agent to return a valid JSON investigation verdict.

json_agent = project_client.agents.create_version(
    agent_name="03-soc-structured-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a SOC analyst. Always respond with ONLY valid JSON "
            "(no markdown fences, no extra text) using this schema:\n"
            '{\n'
            '  "verdict": "AccountCompromised | LikelyCompromised | Suspicious | BenignAnomaly | InsufficientData",\n'
            '  "confidence": "High | Medium | Low",\n'
            '  "risk_factors": ["factor 1", "factor 2", "factor 3"],\n'
            '  "mitre_techniques": ["T1078", "T1110"],\n'
            '  "recommended_actions": ["action 1", "action 2"],\n'
            '  "summary": "2-3 sentence explanation"\n'
            '}'
        ),
    ),
)
r_json = openai_client.responses.create(
    input=(
        "Assess this alert: user anna.kumar@contoso.com received 12 MFA push notifications "
        "in 3 minutes at 10:15 PM from IP 203.0.113.88 (Sao Paulo, BR). She approved the last one. "
        "A new sign-in from that IP followed immediately. Her normal location is Chicago, US."
    ),
    extra_body={"agent_reference": {"name": json_agent.name, "type": "agent_reference"}},
)

print("Raw response:")
print(r_json.output_text)

# Parse and pretty-print
try:
    parsed = json.loads(r_json.output_text)
    print("\n✅ Valid JSON! Parsed fields:")
    for key, value in parsed.items():
        print(f"  {key}: {value}")
except json.JSONDecodeError as e:
    print(f"\n⚠️ JSON parse error: {e}")

Raw response:
{
  "verdict": "LikelyCompromised",
  "confidence": "High",
  "risk_factors": [
    "Multiple rapid MFA push notifications (12 in 3 minutes)",
    "Approval of last MFA prompt from unfamiliar location (Sao Paulo, BR)",
    "Sign-in from IP with geographic deviation from user's usual location (Chicago, US)"
  ],
  "mitre_techniques": [
    "T1078", 
    "T1110",
    "T1531"
  ],
  "recommended_actions": [
    "Immediately initiate password reset and MFA reconfiguration for the affected user",
    "Block or monitor the suspicious IP address 203.0.113.88",
    "Review login and authentication logs for additional suspicious activity",
    "Notify the user and verify recent login activity for any unauthorized access"
  ],
  "summary": "The user received multiple MFA push notifications in rapid succession from a foreign IP address location, and approved the last one leading to an immediate sign-in from that location. This pattern strongly indicates account compromise via MFA fa

In [4]:
# --- 3.3 Few-Shot Prompting ---
# Include example investigation Q&A pairs in the instructions to guide the agent's style.

fewshot_agent = project_client.agents.create_version(
    agent_name="03-soc-fewshot-advisor",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a concise SOC triage advisor. Follow the style shown in these examples:\n\n"
            "USER: Sign-in from NYC and London 8 hours apart, MFA on both, compliant devices, no alerts.\n"
            "ASSISTANT: **Verdict**: BenignAnomaly (High confidence)\n"
            "**Why**: 8-hour gap is consistent with a transatlantic flight (~7-8 hours). "
            "MFA verified on both sign-ins, compliant devices, no post-login anomalies.\n"
            "**Action**: Close alert. No remediation needed.\n\n"
            "USER: Sign-in from Seattle and Lagos 17 minutes apart, second sign-in password-only from malicious IP.\n"
            "ASSISTANT: **Verdict**: AccountCompromised (High confidence)\n"
            "**Why**: 17-minute gap between Seattle and Lagos is physically impossible (~10,000 km). "
            "Second sign-in from a known malicious proxy IP with no MFA. Classic credential theft pattern.\n"
            "**Action**: Revoke sessions, reset password, review MFA methods, check for mailbox rules.\n\n"
            "Always use this exact format: **Verdict**, **Why**, **Action**."
        ),
    ),
)

r_fs = openai_client.responses.create(
    input=(
        "Sign-in from Denver at 3:10 AM and Beijing 20 minutes later. "
        "First sign-in was password+MFA from corporate IP. Second was password-only from a Tor exit node. "
        "User is a Global Administrator."
    ),
    extra_body={"agent_reference": {"name": fewshot_agent.name, "type": "agent_reference"}},
)
print("🧑 Denver → Beijing in 20 min, Global Admin, Tor exit node\n")
print(f"🤖 {r_fs.output_text}")

🧑 Denver → Beijing in 20 min, Global Admin, Tor exit node

🤖 **Verdict**: AccountCompromised (High confidence)  
**Why**: 20-minute gap between Denver and Beijing is physically impossible. First sign-in used MFA from corporate IP, second from a Tor exit node with password-only. High-value Global Administrator account increases risk.  
**Action**: Immediately revoke all sessions, reset password, enforce MFA for all future logins, investigate for lateral movement and data exfiltration, review audit logs.


---
## Section 4: Multi-Step Reasoning & Evaluation (~10 min)

In this section we'll:
1. Build an agent that performs **multi-step reasoning** (analyze → research → assess → recommend)
2. **Evaluate** agent quality using Microsoft's built-in evaluators

In [5]:
# --- 4.1 Multi-Step Reasoning Agent ---
# This agent analyzes a complex security scenario, assesses risk factors,
# and produces a structured investigation report — all in one turn.

investigation_agent = project_client.agents.create_version(
    agent_name="03-soc-investigation-analyst",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a senior SOC L2 analyst. When given a security alert, follow these steps:\n\n"
            "STEP 1 — ALERT PARSING: Identify the user, locations, timestamps, IPs, and alert type.\n"
            "STEP 2 — RISK ASSESSMENT: Evaluate location anomaly, authentication method, IP reputation, "
            "and any post-login indicators.\n"
            "STEP 3 — MITRE MAPPING: Map observed techniques to MITRE ATT&CK framework.\n"
            "STEP 4 — INVESTIGATION REPORT: Produce a structured output with:\n"
            "  - Alert Summary (2-3 sentences)\n"
            "  - Risk Factors Evaluated (numbered list with severity per factor)\n"
            "  - MITRE ATT&CK Techniques (list)\n"
            "  - Verdict: AccountCompromised | LikelyCompromised | Suspicious | BenignAnomaly | InsufficientData\n"
            "  - Confidence: High | Medium | Low\n"
            "  - Recommended Actions (numbered list)\n\n"
            "Show your reasoning at each step. Use markdown formatting."
        ),
        tools=[WebSearchPreviewTool()],
    ),
)

multi_step_query = (
    "Investigate this alert: user tom.wilson@contoso.com (Finance Director) signed in from Berlin, DE "
    "at 10:30 AM using password-only auth. 5 minutes later, an OAuth consent was granted to an app called "
    "'SecureDoc-Viewer' with Mail.Read, Files.ReadWrite.All, and User.Read permissions. The IP 198.51.100.22 "
    "is flagged as suspicious. The user's normal location is Chicago, US. No VPN usage detected."
)

r_ms = openai_client.responses.create(
    input=multi_step_query,
    extra_body={"agent_reference": {"name": investigation_agent.name, "type": "agent_reference"}},
)
print(f"🧑 {multi_step_query[:80]}...\n")
print(f"🤖 {r_ms.output_text}")

# Save for evaluation
multi_step_response = r_ms.output_text

🧑 Investigate this alert: user tom.wilson@contoso.com (Finance Director) signed in...

🤖 ### STEP 1 — ALERT PARSING

- **User:** tom.wilson@contoso.com (Finance Director)
- **Locations:** Sign-in from Berlin, DE; normal location Chicago, US
- **Timestamps:** Sign-in at 10:30 AM; OAuth consent granted 10:35 AM
- **Authentication method:** Password-only authentication
- **IP address:** 198.51.100.22 (flagged as suspicious)
- **Alert type:** Anomalous sign-in from unusual location and OAuth consent granted to an app ('SecureDoc-Viewer') with high permissions (Mail.Read, Files.ReadWrite.All, User.Read)

---

### STEP 2 — RISK ASSESSMENT

1. **Location anomaly:** High severity  
   The user normally signs in from Chicago, US, but signed in from Berlin, DE without VPN usage, indicating possible compromised credentials or session.

2. **Authentication method:** Medium severity  
   Password-only authentication is less secure than MFA methods, increasing risk of compromise.

3. **IP reputation

In [6]:
# --- 4.2 Single-Response Evaluation ---
# Evaluate the agent's output using built-in cloud evaluators via the OpenAI Evals API.

import time
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileContent,
    SourceFileContentContent,
    SourceFileID,
)

data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "response": {"type": "string"},
        },
        "required": ["query", "response"],
    },
)

testing_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "coherence",
        "evaluator_name": "builtin.coherence",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "relevance",
        "evaluator_name": "builtin.relevance",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "fluency",
        "evaluator_name": "builtin.fluency",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
]

eval_obj = openai_client.evals.create(
    name="SOC Investigation Response Evaluation",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)

# Run with inline data (the multi-step response from above)
single_run = openai_client.evals.runs.create(
    eval_id=eval_obj.id,
    name="soc-investigation-check",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileContent(
            type="file_content",
            content=[SourceFileContentContent(item={"query": multi_step_query, "response": multi_step_response})],
        ),
    ),
)

while True:
    single_run = openai_client.evals.runs.retrieve(run_id=single_run.id, eval_id=eval_obj.id)
    if single_run.status in ("completed", "failed"):
        break
    time.sleep(3)

print("Evaluating the SOC investigation response...\n")
if single_run.status == "completed":
    items = list(openai_client.evals.runs.output_items.list(run_id=single_run.id, eval_id=eval_obj.id))
    for r in items[0].results:
        print(f"  {r.name}: {r.score:.0f}/5 ({r.label})")
        if r.reason:
            print(f"    Reason: {r.reason[:120]}...")
        print()
else:
    print(f"❌ Evaluation failed: {single_run.status}")

Evaluating the SOC investigation response...

  coherence: 5/5 (pass)
    Reason: The response is coherent because it logically organizes information from parsing the alert to risk assessment, MITRE map...

  relevance: 5/5 (pass)
    Reason: The response thoroughly analyzes the alert by parsing details, assessing risks, mapping MITRE techniques, and providing ...

  fluency: 4/5 (pass)
    Reason: The response demonstrates proficient fluency with well-structured sentences, appropriate vocabulary, and clear, logical ...



In [7]:
# --- 4.3 Batch Evaluation with Dataset ---
# Run evaluation across multiple pre-built SOC query/response pairs using the cloud Evals API.
# Results are published to the Foundry portal.

# 1. Upload the evaluation dataset
eval_data_path = str(Path.cwd() / "sample_data" / "eval_dataset.jsonl")
data_id = project_client.datasets.upload_file(
    name="soc-eval-data",
    version="1",
    file_path=eval_data_path,
).id
print(f"📄 Dataset uploaded: {data_id}")

# 2. Define schema matching our JSONL fields
batch_data_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "response": {"type": "string"},
            "context": {"type": "string"},
        },
        "required": ["query", "response"],
    },
)

# 3. Define evaluators
batch_criteria = [
    {
        "type": "azure_ai_evaluator",
        "name": "coherence",
        "evaluator_name": "builtin.coherence",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "relevance",
        "evaluator_name": "builtin.relevance",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "fluency",
        "evaluator_name": "builtin.fluency",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {"query": "{{item.query}}", "response": "{{item.response}}"},
    },
    {
        "type": "azure_ai_evaluator",
        "name": "groundedness",
        "evaluator_name": "builtin.groundedness",
        "initialization_parameters": {"deployment_name": MODEL_DEPLOYMENT_NAME},
        "data_mapping": {
            "query": "{{item.query}}",
            "response": "{{item.response}}",
            "context": "{{item.context}}",
        },
    },
]

# 4. Create evaluation + run
batch_eval = openai_client.evals.create(
    name="SOC Alert Triage Batch Evaluation",
    data_source_config=batch_data_config,
    testing_criteria=batch_criteria,
)

print(f"⏳ Running batch evaluation...\n")
batch_run = openai_client.evals.runs.create(
    eval_id=batch_eval.id,
    name="soc-batch-run",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileID(type="file_id", id=data_id),
    ),
)

# 5. Poll for completion
while True:
    batch_run = openai_client.evals.runs.retrieve(run_id=batch_run.id, eval_id=batch_eval.id)
    if batch_run.status in ("completed", "failed"):
        break
    print("  Waiting for eval run to complete...")
    time.sleep(5)

if batch_run.status == "completed":
    output_items = list(
        openai_client.evals.runs.output_items.list(run_id=batch_run.id, eval_id=batch_eval.id)
    )
    print(f"=== Results ({len(output_items)} rows) ===")
    for i, item in enumerate(output_items):
        scores = {r.name: f"{r.score:.0f} ({r.label})" for r in item.results}
        print(f"  Row {i+1}: {scores}")

    print(f"\n🔗 View in Foundry Portal: {batch_run.report_url}")
    print("✅ Batch evaluation complete!")
else:
    print(f"❌ Evaluation failed: {batch_run.status}")

📄 Dataset uploaded: azureai://accounts/fndryiac2ttkx3/projects/iac-ops-demo/data/soc-eval-data/versions/1
⏳ Running batch evaluation...

  Waiting for eval run to complete...
  Waiting for eval run to complete...
  Waiting for eval run to complete...
  Waiting for eval run to complete...
  Waiting for eval run to complete...
  Waiting for eval run to complete...
  Waiting for eval run to complete...
=== Results (6 rows) ===
  Row 1: {'coherence': '4 (pass)', 'relevance': '4 (pass)', 'fluency': '4 (pass)', 'groundedness': '5 (pass)'}
  Row 2: {'coherence': '5 (pass)', 'relevance': '5 (pass)', 'fluency': '4 (pass)', 'groundedness': '5 (pass)'}
  Row 3: {'coherence': '4 (pass)', 'relevance': '4 (pass)', 'fluency': '3 (pass)', 'groundedness': '5 (pass)'}
  Row 4: {'coherence': '4 (pass)', 'relevance': '5 (pass)', 'fluency': '4 (pass)', 'groundedness': '5 (pass)'}
  Row 5: {'coherence': '4 (pass)', 'relevance': '5 (pass)', 'fluency': '4 (pass)', 'groundedness': '5 (pass)'}
  Row 6: {'cohere

> 💡 **View results**: Click the link above to see per-row scores and aggregate metrics
> in the **Foundry portal** (under Build → Evaluation).

---
Proceed to `04-orchestration.ipynb`.